# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [17]:
#imports
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
# load customers.csv into a DataFrame
df = pd.read_csv('./customers.csv')
# take a first look (head, info)
head = df.head(10)
print(f"--------------------------HEAD--------------------------------\n {head}")
info = df.info()
print(info)


--------------------------HEAD--------------------------------
    customer_id  age  income  spending region subscribed
0            1   25    61.3      39.1   West         No
1            2   25    54.9      42.1  South        Yes
2            3   51   105.1      59.9  South         No
3            4   39     NaN      49.9  South         No
4            5   43    72.4      48.3   East         No
5            6   44    84.7      47.1   East         No
6            7   48    92.9      60.0   East        Yes
7            8   21    53.3      36.0  South         No
8            9   39    71.9      48.9   East        Yes
9           10   25    70.4      42.1   east        Yes
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     4

## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [18]:
# HINT 3: one value is impossible for its column -> treat it as missing, then fill 
# (DOING THIS FIRST SO THAT THE NEGATIVE INCOME GETS FILLED WITH THE MEDIAN)
df.loc[df['income'] <= 0, 'income'] = np.nan

# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# Data prior to imputing
def missing_values():
    print("Missing values per column:")
    null_values = df.isna().sum()
    print(null_values.to_string())

    print("\nPercentage of missing values:")
    pct_missing = null_values.sum()/(len(df) * len(df.columns))
    print(f"{pct_missing * 100:.2F}%")

    print("\nNumber of rows WITHOUT any missing values:")
    rows_with_missing = len((df[(df.customer_id.isna()) | (df.income.isna())]))
    print(len(df) - rows_with_missing)

print("\n--------------BEFORE IMPUTING --------------")
missing_values()
df.head(10)

df['income'] = df['income'].fillna(df['income'].median())

# Data after imputing
print("\n--------------AFTER IMPUTING --------------")
missing_values()

# HINT 2: one category label is written in the wrong case -> standardize it

#Finding different formats for regions in the dataset
print("\nDistinct Region Values:", df["region"].nunique())
print(sorted(df["region"].dropna().unique(), reverse=True)[:20])

#Replacing east with East=
df["region"] = df["region"].replace(["east"], "East")

#Checking Work
print("\nDistinct Sex Values After Cleaning:", df["region"].nunique())
print(sorted(df["region"].dropna().unique(), reverse=True)[:20])

print(df.head(30))



--------------BEFORE IMPUTING --------------
Missing values per column:
customer_id    0
age            0
income         3
spending       0
region         0
subscribed     0

Percentage of missing values:
1.25%

Number of rows WITHOUT any missing values:
37

--------------AFTER IMPUTING --------------
Missing values per column:
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0

Percentage of missing values:
0.00%

Number of rows WITHOUT any missing values:
40

Distinct Region Values: 4
['east', 'West', 'South', 'East']

Distinct Sex Values After Cleaning: 3
['West', 'South', 'East']
    customer_id  age  income  spending region subscribed
0             1   25    61.3      39.1   West         No
1             2   25    54.9      42.1  South        Yes
2             3   51   105.1      59.9  South         No
3             4   39    84.7      49.9  South         No
4             5   43    72.4      48.3   East         No
5             6

**Explain each issue:** what you found and how you handled it.

Issue #1 - Missing income values. To solve this problem, I first identified which column has missing values. I then filled the missing values with the median of income in the dataset. I used median to avoid over and under inflation from high or low incomes.

Issue #2 - Category in wrong case. To solve this issue, I first identified which column was in the wrong case. I then converted every instance (east) to the correct case (East).

Issue #3 - Negative income. To solve this issue, I simply made the value NA, then filled it with the median income, just like I did for the other missing income values.

## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [19]:
#a) data types involved: income: float, spending = float
#b) tool: Pearson Correlation
#c) code: 
from scipy.stats import pearsonr
correlation, p_value = pearsonr(df["income"], df["spending"])
print("Pearson correlation:", round(correlation,2))
print("p-value:", round(p_value,4))

Pearson correlation: 0.72
p-value: 0.0


*Your answer:*
Customers who earn more also spend more because the correlation is high and the p-value is low.


### Q2. Is a customer's region related to whether they subscribed?

In [20]:
#a) data types involved: region = str, ssubscribed = string
#b) tool: chi2_contingency to build a contingency table, and pandas crosstab to the place the two variables together.
#c) code:

import pandas as pd
from scipy.stats import chi2_contingency

#cross tab the varibles
table = pd.crosstab(df['region'], df['subscribed'])

# Perform the chi-square test
chi2, p, dof, expected = chi2_contingency(table, correction=False)

# Display the contingency table
print("Contingency Table:")
print(table)

chi_square_statistic = round(chi2, 2)
p_value = round(p,5)

# Display the test results
print(f"\nChi-square statistic: {chi_square_statistic}")
print(f"Degrees of freedom: {dof}")
print(f"P-value (statistical significance): {p_value}")

# Determine statistical significance
if p < 0.05:
    print("The test is statistically significant.")
else:
    print("The test is not statistically significant.")

Contingency Table:
subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Chi-square statistic: 13.16
Degrees of freedom: 2
P-value (statistical significance): 0.00139
The test is statistically significant.


*Your answer:* A customers region is related to whether or not they are subscribed because the chi^2 test results in statistically significant results, so we reject the null hypothesis and conclude that subscription status is related to region.


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [21]:
#a) data types involved: age = int, income = float
#b) tool: statsmodels.api sum() function
#c) code:
X = df[["income", "age"]]
X = sm.add_constant(X)

y = df["spending"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        16:06:48   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:* Yes, spending can be predicted from income and age together, as the model explains 52.0% of the variation in spending (R^2 = 0.520). Income is a significant predictor of spending (p = 0.003), while age is not a significant predictor (p = 0.515).


### Q4. Which region has the most customers, and what is the average income overall?

In [22]:
# hint: count a categorical column
# take the mean of a numeric column
#a) data types involved: region = string, customer_id = int, income = float
#b) tool: Pandas .value_counts and .mean
#c) code:

print(df["region"].value_counts())
avg_income = round(df["income"].mean(), 2)
print(f"Average income: ${avg_income * 1000}")

region
East     21
West     10
South     9
Name: count, dtype: int64
Average income: $86460.0


*Your answer:* The East region has the most customers and the average income is $84,460


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* The specific data types that we had to work with decided which tools needed to be used. 


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* Correlation does not equal causation, so we can not conclude that higher income causes higher spending.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [23]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?
antincipated_df = pd.DataFrame(expected, index = table.index, columns = table.columns)
print(antincipated_df)

print("All expected counts >=5", (expected >=5).all())

subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0
All expected counts >=5 False


*Your answer:* The expected counts are lower than five for some cells, therefore the test is mathematically unstable (sample size is too small), and the company cannot trust the results. 


### Push your code to GitHub